# 28 — Streaming Outlier Rejection

Live-data pipelines hit two universes of bad pixels:

1. **Cosmic rays** — single-frame outliers, ~10⁵ counts in a single pixel
2. **Spatial spikes** — clusters of bright pixels (sample fluorescence, dead pixels lighting up)
3. **Azimuthal sigma-clipping** — per-ring intensity outliers (parasitic spots, broken modules)

`midas_integrate_v2.streaming.outlier` provides all three as
post-frame filters compatible with `integrate_stream`.

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np, torch, matplotlib.pyplot as plt

In [ ]:
from midas_integrate_v2.streaming.outlier import (
    reject_cosmic_rays, reject_spatial_spikes,
    azimuthal_sigma_clip, azimuthal_sigma_clip_multi_panel,
)

# Synthetic frame with cosmic ray hits
img = np.random.default_rng(0).poisson(100, size=(1024, 1024)).astype(np.float32)
# Inject a few cosmics
for _ in range(5):
    iy = np.random.randint(0, 1024); iz = np.random.randint(0, 1024)
    img[iz, iy] += 50000

clean = reject_cosmic_rays(img, threshold_sigma=10.0)
print(f'cosmic rejection: removed {int((img != clean).sum())} pixels')

## Azimuthal sigma-clip for parasitic spots

Multi-panel detectors often pick up parasitic single-crystal spots from
the sample environment (sample-mount Bragg, sample-holder scattering).
These show up as bright points at random η values within a ring.

`azimuthal_sigma_clip(cake_intensity, n_sigma=3.0)` flags eta-bins that
are >3σ above the per-ring median. For multi-panel detectors with
panel-gap artefacts, use the multi-panel variant which masks gap pixels
before computing the sigma threshold.